Our group's GitHub project link: https://github.com/chungite/455-group-project-.git 

Our README contains instructions about how to download the dataset. 

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8, 4)

DATA = Path("csv_mania")          # <-- folder containing the Kaggle csv files
SAMPLE_N = 1_000_000         # rows used for join-heavy plots 
SEED = 42

Reading all the csv files


In [ ]:
customers = pd.read_csv(DATA / "customers.csv")
articles  = pd.read_csv(DATA / "articles.csv")

# shared categorical dtype so joins/groupbys on customer_id are fast and light
cust_dtype = pd.CategoricalDtype(categories=customers["customer_id"].unique())
customers["customer_id"] = customers["customer_id"].astype(cust_dtype)

tx = pd.read_csv(
    DATA / "transactions_train.csv",
    dtype={"customer_id": cust_dtype, "article_id": "int32",
           "price": "float32", "sales_channel_id": "int8"},
    parse_dates=["t_dat"],
)
print(customers.shape, articles.shape, tx.shape)

Looking at the dataset structure


In [ ]:
def overview(df, name):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    print(f"== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols | "
          f"{df.memory_usage(deep=True).sum() / 1e6:,.0f} MB | duplicated rows: {df.duplicated().sum():,}")
    return out

overview(customers, "customers")
overview(articles, "articles")
overview(tx, "transactions")

In [ ]:
print(overview(tx.sample(2_000_000, random_state=SEED), "transactions (2M-row sample for speed)"))
print("\nDate range:", tx["t_dat"].min().date(), "->", tx["t_dat"].max().date())
tx.head()

In [ ]:
# Coverage / referential integrity
print("Customers with >=1 transaction:", f"{tx['customer_id'].nunique():,} / {len(customers):,}",
      f"({tx['customer_id'].nunique() / len(customers):.1%})")
print("Articles sold at least once  :", f"{tx['article_id'].nunique():,} / {len(articles):,}",
      f"({tx['article_id'].nunique() / len(articles):.1%})")
print("Transaction article_ids missing from articles.csv:", (~tx['article_id'].drop_duplicates().isin(articles['article_id'])).sum())

Working with customers csv


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(customers["age"].dropna(), bins=60, ax=ax[0])
ax[0].set_title("Customer age distribution")
sns.boxplot(x=customers["age"].dropna(), ax=ax[1])
ax[1].set_title("Customer age (box plot)")
plt.tight_layout(); plt.show()

print(customers["age"].describe().round(1))
print("\nAge < 18:", (customers["age"] < 18).sum(), "| Age > 80:", (customers["age"] > 80).sum(),
      "| missing:", customers["age"].isna().sum())

In [ ]:
# dropna=False because the missing values are informative here
cat_cols = ["club_member_status", "fashion_news_frequency"]
fig, axes = plt.subplots(1, len(cat_cols), figsize=(8, 3.8))
for a, c in zip(axes, cat_cols):
    vc = customers[c].value_counts(dropna=False)
    sns.barplot(x=vc.index.astype(str), y=vc.values, ax=a)
    a.set_title(c); a.set_xlabel(""); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

for c in cat_cols:
    print(customers[c].value_counts(dropna=False, normalize=True).round(3).to_dict())

working with articles.csv


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for a, c in zip(axes.ravel(), ["index_group_name", "product_group_name", "garment_group_name",
                               "product_type_name", "colour_group_name", "graphical_appearance_name"]):
    vc = articles[c].value_counts().head(15)
    sns.barplot(x=vc.values, y=vc.index, ax=a, color="steelblue")
    a.set_title(f"{c} (top 15, of {articles[c].nunique()})"); a.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
# Missing text + description length
print("Missing detail_desc:", articles["detail_desc"].isna().sum())
articles["desc_len"] = articles["detail_desc"].fillna("").str.len()
sns.histplot(articles["desc_len"], bins=60); plt.title("detail_desc length (characters)"); plt.show()

# Redundant code/name column pairs (1:1 mapping => duplicated information)
pairs = [c[:-5] for c in articles.columns if c.endswith("_name") and (c[:-5] + "_no" in articles.columns or c[:-5] + "_code" in articles.columns)]
for p in pairs:
    code_col = p + "_no" if p + "_no" in articles.columns else p + "_code"
    print(f"{p:28s} unique codes={articles[code_col].nunique():4d}  unique names={articles[p + '_name'].nunique():4d}")

working with Transactions.csv


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(tx["price"].sample(2_000_000, random_state=SEED), bins=100, ax=ax[0])
ax[0].set_title("Price (raw)")
sns.histplot(np.log10(tx["price"].sample(2_000_000, random_state=SEED)), bins=100, ax=ax[1])
ax[1].set_title("log10(price)")
sns.boxplot(data=tx.sample(1_000_000, random_state=SEED), x="sales_channel_id", y="price", ax=ax[2])
ax[2].set_yscale("log"); ax[2].set_title("Price by sales channel (log y)")
plt.tight_layout(); plt.show()

print(tx["price"].describe().round(4))
print("\nSales channel share:\n", tx["sales_channel_id"].value_counts(normalize=True).round(3))

In [ ]:
daily = tx.groupby("t_dat").size()
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
daily.plot(ax=ax[0]); ax[0].set_title("Transactions per day")
dow = tx["t_dat"].dt.day_name().value_counts().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
sns.barplot(x=dow.index, y=dow.values, ax=ax[1]); ax[1].tick_params(axis="x", rotation=45); ax[1].set_title("By weekday")
tx.groupby(tx["t_dat"].dt.to_period("M")).size().plot(kind="bar", ax=ax[2]); ax[2].set_title("By month")
plt.tight_layout(); plt.show()

In [ ]:
# Long tails: purchases per customer and per article
per_cust = tx.groupby("customer_id", observed=True).size()
per_art  = tx.groupby("article_id").size()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for axis, values, title in [(ax[0], per_cust, "Purchases per customer"), (ax[1], per_art, "Purchases per article")]:
    bins = np.geomspace(values.min(), values.max(), 80)
    sns.histplot(values, bins=bins, ax=axis)
    axis.set_xscale("log"); axis.set_yscale("log")
    axis.set_title(f"{title} (log-log)")
    axis.set_xlabel("Purchase count (log scale)")
    axis.set_ylabel("Number of customers/articles (log scale)")
plt.tight_layout(); plt.show()

print(per_cust.describe().round(1))
print("\nCustomers with exactly 1 purchase:", f"{(per_cust == 1).mean():.1%}",
      "| <=5 purchases:", f"{(per_cust <= 5).mean():.1%}")

# Popularity concentration
s = per_art.sort_values(ascending=False)
cum = s.cumsum() / s.sum()
plt.plot(np.arange(1, len(cum) + 1) / len(articles), cum.values)
plt.xlabel("fraction of articles (most popular first)"); plt.ylabel("cumulative share of sales")
plt.title("Popularity concentration"); plt.show()
for k in (100, 1000, 10000):
    print(f"Top {k:>6,} articles = {cum.iloc[k-1]:.1%} of all transactions")

In [ ]:
# Sparsity of the user-item matrix + repeat purchases (on a 5% customer subset to save memory)
n_cust, n_art = tx["customer_id"].nunique(), tx["article_id"].nunique()
pairs_n = len(tx[["customer_id", "article_id"]].drop_duplicates())
print(f"User-item matrix density: {pairs_n / (len(customers) * len(articles)):.5%}")

sub = tx[tx["customer_id"].cat.codes % 20 == 0]
dup_share = sub.duplicated(["customer_id", "article_id"]).mean()
print(f"Share of transactions that re-buy an article the customer already bought (5% subset): {dup_share:.1%}")

Testing random relationships and correlations between the files


1) Joining a random sample of transactions with customer and article attributes to create a relationship plot


In [ ]:
df = (tx.sample(SAMPLE_N, random_state=SEED)
        .merge(customers, on="customer_id", how="left")
        .merge(articles.drop(columns="desc_len"), on="article_id", how="left"))
df["age_bin"] = pd.cut(df["age"], [0, 19, 24, 29, 34, 44, 54, 64, 100],
                       labels=["<20", "20-24", "25-29", "30-34", "35-44", "45-54", "55-64", "65+"])
df.shape

In [ ]:
# . What do different age groups buy? (row-normalised: each age bin sums to 1)
ct = pd.crosstab(df["age_bin"], df["index_group_name"], normalize="index")
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues")
plt.title("Share of purchases by index group, per age bin"); plt.show()

ct2 = pd.crosstab(df["sales_channel_id"], df["index_group_name"], normalize="index")
sns.heatmap(ct2, annot=True, fmt=".2f", cmap="Blues"); plt.title("Index group mix by sales channel"); plt.show()

In [ ]:
# Seasonality by product group (weekly sales, top 6 groups)
top = tx.merge(articles[["article_id", "product_group_name"]], on="article_id")
top_groups = top["product_group_name"].value_counts().head(6).index
wk = (top[top["product_group_name"].isin(top_groups)]
      .groupby([pd.Grouper(key="t_dat", freq="W"), "product_group_name"]).size().unstack())
wk.plot(figsize=(12, 4)); plt.title("Weekly transactions by product group"); plt.show()
del top

In [ ]:
# Numeric correlation heatmap at the CUSTOMER level (Spearman: data is heavily skewed)
tx["is_online"] = (tx["sales_channel_id"] == 2).astype("int8")
agg = tx.groupby("customer_id", observed=True).agg(
    n_purchases=("article_id", "size"),
    n_unique_articles=("article_id", "nunique"),
    total_spend=("price", "sum"),
    avg_price=("price", "mean"),
    online_share=("is_online", "mean"),
)
agg = agg.join(customers.set_index("customer_id")[["age"]])

corr = agg.corr(method="spearman")
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Spearman correlation: customer-level features"); plt.show()

In [ ]:
#  Association between categorical article attributes (Cramer's V; 0 = none, 1 = identical info)
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = ((ct - np.outer(ct.sum(1), ct.sum(0)) / ct.values.sum()) ** 2 /
            (np.outer(ct.sum(1), ct.sum(0)) / ct.values.sum())).values.sum()
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_, k_ = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / max(1e-12, min(k_ - 1, r_ - 1)))

acols = ["index_group_name", "index_name", "product_group_name", "product_type_name",
         "garment_group_name", "section_name", "department_name",
         "colour_group_name", "perceived_colour_master_name", "graphical_appearance_name"]
V = pd.DataFrame(index=acols, columns=acols, dtype=float)
for a in acols:
    for b in acols:
        V.loc[a, b] = 1.0 if a == b else cramers_v(articles[a], articles[b])

plt.figure(figsize=(9, 7))
sns.heatmap(V, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title("Cramer's V between article attributes"); plt.show()

In [ ]:
#  Price vs. product group, and price vs. age
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
order = df.groupby("product_group_name")["price"].median().sort_values().index
sns.boxplot(data=df, x="product_group_name", y="price", order=order, ax=ax[0], showfliers=False)
ax[0].tick_params(axis="x", rotation=60); ax[0].set_title("Price by product group (outliers hidden)")
sns.boxplot(data=df, x="age_bin", y="price", ax=ax[1], showfliers=False); ax[1].set_title("Price by age bin")
plt.tight_layout(); plt.show()

In [ ]:
# Article popularity across the full history and latest 12 months
# Exploratory counts include unsold articles; ties remain together in tertile estimates.
end_date = tx["t_dat"].max()
window_starts = {
    "Full history": tx["t_dat"].min(),
    "Latest 12 months": end_date - pd.DateOffset(years=1),
}
article_features = ["product_group_name", "index_group_name", "colour_group_name"]
popularity_by_window = {}
summary_rows = []
class_rows = []

for window_name, start_date in window_starts.items():
    window_tx = tx.loc[tx["t_dat"] >= start_date]
    purchase_counts = (
        window_tx.groupby("article_id").size()
        .reindex(articles["article_id"], fill_value=0)
        .to_numpy(dtype="int64")
    )
    article_pop = articles[["article_id", *article_features]].copy()
    article_pop["purchase_count"] = purchase_counts
    popularity_by_window[window_name] = article_pop

    q1, q2 = pd.Series(purchase_counts).quantile([1 / 3, 2 / 3]).to_numpy()
    class_counts = {
        "Low (<= first cutoff)": int((purchase_counts <= q1).sum()),
        "Medium (between cutoffs)": int(((purchase_counts > q1) & (purchase_counts <= q2)).sum()),
        "High (> second cutoff)": int((purchase_counts > q2).sum()),
    }
    class_rows.append({"window": window_name, "q1 cutoff": q1, "q2 cutoff": q2, **class_counts})
    summary_rows.append({
        "window": window_name,
        "start": start_date.date(),
        "end": end_date.date(),
        "articles": len(purchase_counts),
        "unsold articles": int((purchase_counts == 0).sum()),
        "unsold %": 100 * (purchase_counts == 0).mean(),
        "median purchases/article": float(np.median(purchase_counts)),
        "90th percentile": float(np.quantile(purchase_counts, 0.90)),
        "99th percentile": float(np.quantile(purchase_counts, 0.99)),
        "maximum purchases": int(purchase_counts.max()),
    })

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (window_name, article_pop) in zip(axes, popularity_by_window.items()):
    sns.histplot(np.log1p(article_pop["purchase_count"]), bins=60, ax=ax)
    ax.set_title(f"Article purchase counts: {window_name}")
    ax.set_xlabel("log(1 + purchases per article)")
    ax.set_ylabel("Number of articles")
plt.tight_layout(); plt.show()

print("Popularity distribution (zero-sales articles included):")
print(pd.DataFrame(summary_rows).to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
print("\nTie-aware approximate tertiles (equal purchase counts are not split):")
print(pd.DataFrame(class_rows).to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

fig, axes = plt.subplots(2, len(article_features), figsize=(18, 9))
for row, (window_name, article_pop) in enumerate(popularity_by_window.items()):
    for col, feature in enumerate(article_features):
        grouped = article_pop.groupby(feature, observed=True).agg(
            articles=("article_id", "size"),
            pct_sold=("purchase_count", lambda values: 100 * (values > 0).mean()),
        ).reset_index()
        # Limit the plot to common categories to keep labels readable.
        plot_data = grouped.nlargest(12, "articles").sort_values("pct_sold")
        ax = axes[row, col]
        sns.barplot(data=plot_data, x="pct_sold", y=feature, ax=ax, color="steelblue")
        ax.set_title(f"{window_name}: sold share by {feature}")
        ax.set_xlabel("Articles with at least one purchase (%)")
        ax.set_ylabel("")
plt.tight_layout(); plt.show()


## Key insights and challenges

The three tables cover 1.37 million customers, 105,542 articles, and 31.8 million transactions from 2018-09-20 to 2020-09-22. Coverage is strong: 99.3% of customers and 99.1% of articles appear in transactions, with no orphan article IDs, although 995 articles were never sold. Purchases are highly uneven: the median customer made 9 purchases (versus a mean of 23.3), the top 10,000 articles account for 59.6% of transactions, and the customer-item matrix is only 0.01886% dense; 14.3% of transactions in the sampled subset are repeat customer-article purchases, including possible same-day repeats. Customers have a median age of 32, and Ladieswear dominates purchases across age groups, while Divided's share declines with age; product mix differs only modestly by sales channel. Sales also show clear seasonality, especially for swimwear, which peaks in summer, alongside a few sharp volume spikes that may reflect promotions. Customer-level categorical summaries cover club membership status and fashion-news frequency, and the customer-level correlation analysis retains age but excludes the FN and Active flags. Some article features are redundant because of nested product classifications, while customer age is largely unrelated to purchase volume. Important challenges are the transaction file's scale, roughly 3.0 million exact duplicate rows that should not be removed without deciding whether they represent separate units, sparse and short customer histories, missing customer-profile values, the exclusion of FN and Active from this EDA (which limits conclusions about those engagement indicators), anomalous zero prices and high-cardinality or free-text product attributes. Popularity-focused EDA compares article purchase counts across the full history and latest 12 months, includes zero-sales articles, reports tie-aware approximate tertile sizes, and compares sold shares by common product, index, and colour categories. Final popularity labels remain a separate preprocessing decision. Popularity labels will depend on the chosen time window and can be distorted by seasonality and spikes; article-level models therefore need carefully aggregated features, leakage-safe feature and label periods, explicit handling of unsold/tied articles, and attention to redundant features. Joined analyses based on a one-million-row sample are exploratory and should be validated before drawing definitive conclusions.

# Preprocessing

## Dataset Cleaning and Additional Feature Extraction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


df = pd.read_csv(
    'csv_mania/articles.csv',
    na_values=['n/a', '--', 'Unknown']
)

# Remove duplicate rows
df.drop_duplicates(inplace=True)

columns_to_check = [
    'graphical_appearance_no',
    'colour_group_code',
    'perceived_colour_value_id',
    'perceived_colour_master_id'
]

# Remove rows where any specified column has a value of -1
rows_with_missing_values = df[df.isna().any(axis=1)].copy()
rows_with_missing_values.to_csv('missing_report.csv', index=False)

#df = df[~(df[columns_to_check] == -1).any(axis=1)].copy()
# Map description keywords to garment groups
keyword_mapping = {
    'denim': {
        'garment_group_name': 'Trousers Denim',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
        'jeans': {
        'garment_group_name': 'Trousers Denim',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
        'trousers': {
        'garment_group_name': 'Trousers',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
            'pants': {
        'garment_group_name': 'Trousers',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
            'joggers': {
        'garment_group_name': 'Trousers',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
            'jacket': {
        'garment_group_name': 'Outdoor',
        'product_type_name': 'Jacket', 
        'product_type_no' : 262,
        'product_group_name' :'Garment Upper body',
    },
                'coat': {
        'garment_group_name': 'Outdoor',
        'product_type_name': 'Jacket', 
        'product_type_no' : 262,
        'product_group_name' :'Garment Upper body',
    },
            'parka': {
        'garment_group_name': 'Outdoor',
        'product_type_name': 'Jacket', 
        'product_type_no' : 262,
        'product_group_name' :'Garment Upper body',
    },
            'skirts': {
        'garment_group_name': 'Skirts',
        'product_type_name': 'Skirt', 
        'product_type_no' : 275,
        'product_group_name' :'Garment Lower body',
    },
               'shorts': {
        'garment_group_name': 'Shorts',
        'product_type_name': 'Shorts', 
        'product_type_no' : 274,
        'product_group_name' :'Garment Lower body',
    },    
               'cardigan': {
        'garment_group_name': 'Knitwear',
        'product_type_name': 'Sweater', 
        'product_type_no' : 252 ,
        'product_group_name' :'Garment Upp body',
    },
               'sweater': {
        'garment_group_name': 'Knitwear',
        'product_type_name': 'Sweater', 
        'product_type_no' : 252 ,
        'product_group_name' :'Garment Upper body',
    },
               'jumper': {
        'garment_group_name': 'Knitwear',
        'product_type_name': 'Sweater', 
        'product_type_no' : 252 ,
        'product_group_name' :'Garment Upper body',
    },
               'knit': {
        'garment_group_name': 'Knitwear',
        'product_type_name': 'Sweater', 
        'product_type_no' : 252 ,
        'product_group_name' :'Garment Upper body',
    },
               'blouse': {
        'garment_group_name': 'Blouse',
        'product_type_name': 'Blouse', 
        'product_type_no' : 258 ,
        'product_group_name' :'Garment Upper body',
    },
                   'T-shirt': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'T-shirt', 
        'product_type_no' : 255 ,
        'product_group_name' :'Garment Upper body',
    },
                   'shirt': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'Shirt', 
        'product_type_no' : 255 ,
        'product_group_name' :'Garment Upper body',
    },
                 'shirt': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'Shirt', 
        'product_type_no' : 259 ,
        'product_group_name' :'Garment Upper body',
    },

                 'jersey': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'Shirt', 
        'product_type_no' : 259 ,
        'product_group_name' :'Garment Upper body',
    },
                 'shirt': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'Shirt', 
        'product_type_no' : 259 ,
        'product_group_name' :'Garment Upper body',
    },
    
                 'sweatshirt': {
        'garment_group_name': 'Jersey Fancy',
        'product_type_name': 'Shirt', 
        'product_type_no' : 259 ,
        'product_group_name' :'Garment Upper body',
    },
                 'v-neck': {
        'garment_group_name': 'Jersey Fancy',
        'product_type_name': 'Top', 
        'product_type_no' : 254 ,
        'product_group_name' :'Garment Upper body',
    },

                     'top': {
        'garment_group_name': 'Jersey Basic',
        'product_type_name': 'Top', 
        'product_type_no' : 254 ,
        'product_group_name' :'Garment Upper body',
    },
                     'dress': {
        'garment_group_name': 'Dresses Ladies',
        'product_type_name': 'Dress', 
        'product_type_no' : 265 ,
        'product_group_name' :'Garment Full body',
    },
                      'bra': {
        'garment_group_name': 'Under-, Nightwear',
        'product_type_name': 'Bra', 
        'product_type_no' : 306 ,
        'product_group_name' :'Garment Upper body',
    },   
                      'briefs': {
        'garment_group_name': 'Under-, Nightwear',
        'product_type_name': 'Underwear bottom', 
        'product_type_no' : 286 ,
        'product_group_name' :'Garment Lower body',
    },     
                      'hipster': {
        'garment_group_name': 'Under-, Nightwear',
        'product_type_name': 'Underwear bottom', 
        'product_type_no' : 286 ,
        'product_group_name' :'Garment Lower body',
    },       
                'tights': {
        'garment_group_name': 'Socks and Tights',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
                'socks': {
        'garment_group_name': 'Socks and Tights',
        'product_type_name': 'Trousers', 
        'product_type_no' : 272,
        'product_group_name' :'Garment Lower body',
    },
                'shoes': {
        'garment_group_name': 'Shoes',
        'product_type_name': 'Flat Shoe', 
        'product_type_no' : 144,
        'product_group_name' :'Shoes',
    },    
                'sandals': {
        'garment_group_name': 'Shoes',
        'product_type_name': 'Flat Shoe', 
        'product_type_no' : 144,
        'product_group_name' :'Shoes',
    },   
                    'hat': {
        'garment_group_name': 'Accessories',
        'product_type_name': 'Hat/beanie', 
        'product_type_no' : 75,
        'product_group_name' :'Accessories',
    },      

                         'patch': {
        'garment_group_name': 'Accessories',
        'product_type_name': ' Other accessories', 
        'product_type_no' : 78,
        'product_group_name' :'Accessories',
    },           
  
                         'band': {
        'garment_group_name': 'Accessories',
        'product_type_name': ' Other accessories', 
        'product_type_no' : 78,
        'product_group_name' :'Accessories',
    }, 
                         'case': {
        'garment_group_name': 'Small Accessories',
        'product_type_name': ' Other accessories', 
        'product_type_no' : 78,
        'product_group_name' :'Accessories',
    }, 
                         'longjohns': {
        'garment_group_name': 'Accessories',
        'product_type_name': ' Other accessories', 
        'product_type_no' : 78,
        'product_group_name' :'Accessories',
    }, 
                         'bag': {
        'garment_group_name': 'Accessories',
        'product_type_name': ' Other accessories', 
        'product_type_no' : 78,
        'product_group_name' :'Accessories',
    }, 
}

def infer_garment_group(row):
    columns_to_check = ['product_type_no']

    if pd.isna(row['detail_desc']):
        return row

    desc = str(row['detail_desc']).lower()

    for keyword, replacements in keyword_mapping.items():
        if keyword.lower() in desc:
            for column, value in replacements.items():
                if (
                    pd.isna(row[column])
                    or (
                        column in columns_to_check
                        and row[column] == -1
                    )
                ):
                    row[column] = value

            break

    return row


# Record missing values before fixing
missing_before = df['garment_group_name'].isna()

# Fill only missing garment_group_name values
df = df.apply(infer_garment_group, axis=1)
df_cleaned = df.dropna(subset=['detail_desc'])
df_cleaned = df_cleaned.dropna(subset=['garment_group_name'])
# Save only rows where garment_group_name was fixed
fixed_rows = df[
    missing_before & df['garment_group_name'].notna()
].copy()


# Save the full dataset with only garment_group_name changed
df_cleaned.to_csv('cleaned_article.csv', index=False)


customers = pd.read_csv('csv_mania/customers.csv')
customers.drop_duplicates(inplace=True)
# 2. Drop the column by name
# Use inplace=True to modify the existing DataFrame directly
customers.drop(columns=['FN', 'Active','postal_code'], inplace=True)

customers["club_member_status"] = (
    customers["club_member_status"].fillna("Missing")
)

customers["fashion_news_frequency"] = (
    customers["fashion_news_frequency"].fillna("Missing")
)

# Impute missing ages with the median
customers["age"] = customers["age"].fillna(
    customers["age"].median()
)

# Verify that missing values are handled
print(customers[["age", "club_member_status",
                 "fashion_news_frequency"]].isna().sum())



customers.to_csv("cleaned_customer.csv", index=False)




df = pd.read_csv('csv_mania/transactions_train.csv')
# Convert the t_dat column to datetime, keeping the DataFrame
df['t_dat'] = pd.to_datetime(df['t_dat'], errors='coerce')

# Save the cleaned dataset
df.to_csv('cleaned_transactions_training.csv', index=False)

# 1. Load article and customer information
articles = pd.read_csv("cleaned_article.csv")
customers = pd.read_csv("cleaned_customer.csv")

# Keep the customer columns needed for feature engineering
customer_columns = [
    "customer_id",
    "age",
    "club_member_status",
    "fashion_news_frequency"
]
customers = customers[customer_columns].copy()

# 2. Read transaction columns
transactions = pd.read_csv(
    "cleaned_transactions_training.csv",
    usecols=[
        "article_id",
        "customer_id",
        "price",
        "sales_channel_id"
    ],
    dtype={
        "article_id": "int32",
        "customer_id": "category",
        "sales_channel_id": "int8"
    }
)

# Make customer ID types consistent before joining
transactions["customer_id"] = transactions["customer_id"].astype(str)
customers["customer_id"] = customers["customer_id"].astype(str)

# 3. Aggregate transaction features by article
transaction_features = transactions.groupby(
    "article_id",
    observed=True
).agg(
    purchase_count=("article_id", "size"),
    unique_customers=("customer_id", "nunique"),
    average_price=("price", "mean"),
    channel_1_share=("sales_channel_id", lambda s: (s == 1).mean()),
    channel_2_share=("sales_channel_id", lambda s: (s == 2).mean())
).reset_index()

# 4. Join customers to transactions using customer_id
customer_transactions = transactions[
    ["article_id", "customer_id"]
].drop_duplicates()

customer_transactions = customer_transactions.merge(
    customers,
    on="customer_id",
    how="left"
)

# 5. Create customer features for each article
# Each customer counts once per article.
customer_features = customer_transactions.groupby(
    "article_id",
    observed=True
).agg(
    average_buyer_age=("age", "mean")
).reset_index()

# Calculate the proportion of buyers in each customer category
membership_shares = pd.crosstab(
    customer_transactions["article_id"],
    customer_transactions["club_member_status"],
    normalize="index"
)
membership_shares.columns = [
    f"club_member_share_{col}" for col in membership_shares.columns
]
membership_shares = membership_shares.reset_index()

news_shares = pd.crosstab(
    customer_transactions["article_id"],
    customer_transactions["fashion_news_frequency"],
    normalize="index"
)
news_shares.columns = [
    f"fashion_news_share_{col}" for col in news_shares.columns
]
news_shares = news_shares.reset_index()

customer_features = customer_features.merge(
    membership_shares, on="article_id", how="left"
).merge(
    news_shares, on="article_id", how="left"
)

del transactions, customer_transactions, customers

# 6. Combine article, transaction, and customer features
df = articles.merge(
    transaction_features,
    on="article_id",
    how="left"
).merge(
    customer_features,
    on="article_id",
    how="left"
)

# Fill transaction features for articles never purchased
zero_features = [
    "unique_customers",
    "average_price",
    "channel_1_share",
    "channel_2_share"
]
df[zero_features] = df[zero_features].fillna(0)
df["never_purchased"] = df["purchase_count"].isna().astype(int)
df[["purchase_count", "unique_customers"]] = df[["purchase_count", "unique_customers"]].fillna(0)
df["channel_1_share"] = df["channel_1_share"].fillna(0)
df = df.drop(columns=["channel_2_share"], errors="ignore")


## Normalization and Standardization

In [ ]:
##### Normalization of numerical features and one hot encoding for categorical features ######

share_cols = [c for c in df.columns
              if c.startswith(("channel_1_share", "club_member_share_", "fashion_news_share_"))]
df[share_cols] = df[share_cols].fillna(0)
df = df.drop(columns=["channel_2_share"], errors="ignore")   # redundant with channel_1_share

# average_price / average_buyer_age stay NaN for unpurchased articles -> imputed in the pipeline
# (make sure you remove average_price from any fillna(0) step)
#save initial purchase count for mutual info 
df["purchase_count_raw"] = df["purchase_count"]
# --- Outliers ---
for col in ["average_price", "average_buyer_age"]:
    lo, hi = df[col].quantile([0.01, 0.99])
    df[col] = df[col].clip(lo, hi)

for col in ["purchase_count", "unique_customers"]:
    df[col] = np.log1p(df[col])
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

# --- Features ---
numeric_features = [ "purchase_count", "unique_customers","average_price", "average_buyer_age"]
passthrough_features = share_cols + ["never_purchased"]   # already in [0, 1]

# 8. Choose categorical article features
categorical_features = [
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name",
    "graphical_appearance_name"
]


X = df.set_index("article_id")[categorical_features + numeric_features + passthrough_features]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]), categorical_features),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("pass", "passthrough", passthrough_features),
])

X_encoded = preprocessor.fit_transform(X)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),  # dense output
        ]), categorical_features),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("pass", "passthrough", passthrough_features),
    ],
    remainder="drop",
    sparse_threshold=0,              # force a dense result
    verbose_feature_names_out=False, # cleaner column names (no "cat__" prefixes)
)

# Return a pandas DataFrame (keeps the article_id index and column names)
preprocessor.set_output(transform="pandas")

X_encoded = preprocessor.fit_transform(X)

print("Original feature columns:", X.shape[1])
print("Encoded feature columns:", X_encoded.shape[1])

X_encoded = X_encoded.astype("float32")

X_encoded.to_csv("encoded_features.csv", index_label="article_id")

## Feature Selection Using Mutual Information 

In [ ]:
######## Mutual information for feature selection #########

# 7. Define popularity category


df["purchase_category"] = pd.cut(
    df["purchase_count_raw"],
    bins=[-1, 0, 24, 173, float("inf")],
    labels=["Never purchased", "Low", "Medium", "High"]
)
print(df["purchase_category"].value_counts())   # sanity check: all 4 classes should appear


# 8. Choose categorical article features


categorical_features = [
    c for c in categorical_features if c in df.columns
]

# Customer categories are represented by proportions,
# so they can be treated as numerical features.

# Remove duplicate feature names and keep available columns

all_numeric_features = [
    "average_price",
    "average_buyer_age"
] + [
    c for c in passthrough_features
    if c != "never_purchased"
]
# 9. Encode article categories
X_cat = df[categorical_features].fillna("Missing").astype(str)

encoder = OrdinalEncoder()
X_cat_encoded = pd.DataFrame(
    encoder.fit_transform(X_cat),
    columns=categorical_features,
    index=df.index
)

X_num = df[all_numeric_features].copy()

# Median-impute missing numerical values
X_num = X_num.fillna(X_num.median()).fillna(0)

X = pd.concat([X_cat_encoded, X_num], axis=1)
y = df["purchase_category"]

valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

# Encoded article categories are discrete.
# Numerical features, including category proportions, are continuous
# for this MI calculation.
mi_numeric = X_num.columns.tolist()

discrete_mask = (
    [True] * len(categorical_features)
    + [False] * len(mi_numeric)
)

assert len(discrete_mask) == X.shape[1]
# 10. Calculate mutual information
mi_scores = mutual_info_classif(
    X,
    y,
    discrete_features=discrete_mask,
    random_state=42
)

scores = pd.Series(
    mi_scores,
    index=X.columns
).sort_values(ascending=False)

print("Mutual Information scores:")
print(scores)

print("\nTop 10 features:")
print(scores.head(10))

scores.to_csv(
    "mutual_information_scores.csv",
    header=["mi_score"]
)

## Dimensionality Reduction

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load the encoded dataset
X = pd.read_csv(
    "encoded_features.csv",
    index_col="article_id"
)

print("Original shape:", X.shape)

# PCA with 95% explained variance
pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X)

print("Original number of features:", X.shape[1])
print("Number of principal components:", X_pca.shape[1])
print("Total explained variance:",
      round(sum(pca.explained_variance_ratio_) * 100, 2), "%")


In [ ]:

# Cumulative explained variance
cumulative_variance = np.cumsum(
    pca.explained_variance_ratio_
)

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance
)

plt.axhline(
    y=0.95,
    color="red",
    linestyle="--",
    label="95% variance"
)

plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance - H&M Dataset")

plt.legend()
plt.grid(True)
plt.show()


### Discussion of Dimensionality Reduction Methods

We decided to use PCA as our dimensionality reduction technique. We used PCA instead of t-SNE because for our project we want to reduce the number of features for classification rather than visualizing the dataset. Also, PCA is usually more computationally efficient. Our dataset is very large, so we needed a technique that could process it efficiently. 

Earlier, we created a combined and processed dataset out of the Articles, Customers, and Transactions tables. We applied One-hot encoding and standardization to this combined dataset, which we then named encoded_features.csv. 

After we applied one-hot encoding, this dataset had 104,886 articles and 607 features. Then, once we applied PCA to it, it took the 607 features and reduced them to 155 principal components. This is a 74.5% reduction in dimensionality, and it preserved 95% of the total variance. 

Looking at the PCA explained variance graph, we can see that the curve starts to level out as more components are added. This means that each additional component captures less of the dataset's variation, suggesting that the earlier components have already captured most of the variation. It also suggests that some of the original features contain overlapping information. So, by reducing the number of features, PCA may help improve the computational efficiency when we train the Random Forest, SVM, and k-NN classification models. 